# 02 — Knowledge Extraction

Turns the raw corpus into structured knowledge the pipeline can reason about.

**Why this step?** The model needs three things to learn ARO well:
1. **Action metadata** — every action's verbs, prepositions, and semantic role,
   extracted directly from Swift source. This powers system prompts and training pairs
   in every downstream notebook.
2. **Q&A seeds** — section headings and code blocks from the Proposals, which become
   the raw material for LLM-generated training pairs in notebook 04.
3. **Working examples** — 103 real ARO programs with expected output, used as
   ground-truth code in generation and validation.

All 61 actions are extracted by scanning `Sources/ARORuntime/Actions/` recursively
(many files define multiple actions — a single-struct-per-file assumption would miss half of them).

**Input:**  `../data/01_corpus/manifest.json`
**Output:** `../data/02_knowledge/knowledge.json`

In [ ]:
import sys, importlib
from pathlib import Path
_cfg_dir = Path('.').resolve()
if _cfg_dir not in [Path(p) for p in sys.path]:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *
print(f'Config loaded | model={MODEL_ID}')
print(f'ARO root:  {ARO_ROOT}')
print(f'Pairs:     {PAIRS_FILE}')
print(f'Adapters:  {ADAPTER_DIR}')

import json
import re
from pathlib import Path
from collections import defaultdict

DATA_OUT = DATA_ROOT / '02_knowledge'
DATA_OUT.mkdir(parents=True, exist_ok=True)

with open(DATA_IN / 'manifest.json') as f:
    corpus = json.load(f)

print(f'Loaded manifest: {len(corpus["aro_files"])} .aro files, '
      f'{len(corpus["proposals"])} proposals, {len(corpus["swift_actions"])} Swift files')

## Extract action metadata from Swift source

In [ ]:
def extract_all_actions_from_swift(swift_path):
    """
    Extract ALL ActionImplementation conformances from one Swift file.
    A single file like FileActions.swift may define 5+ actions.
    Returns a list of dicts; empty if no ActionImplementation found.
    """
    content = Path(swift_path).read_text()
    results = []

    # Match every struct/class that lists ActionImplementation in its conformance chain
    impl_re = re.compile(
        r'(?:public\s+)?(?:struct|class)\s+(\w+)\s*:[^{]+ActionImplementation[^{]*\{'
    )
    for m in impl_re.finditer(content):
        struct_name = m.group(1)
        block_start = m.start()
        # Take a generous window after the struct header for the static members
        block = content[block_start: block_start + 5000]

        verbs_m = re.search(r'static\s+let\s+verbs[^=]*=\s*\[([^\]]+)\]', block, re.DOTALL)
        role_m  = re.search(r'static\s+let\s+role[^=]*=\s*\.(\w+)', block)
        prep_m  = re.search(r'static\s+let\s+validPrepositions[^=]*=\s*\[([^\]]+)\]', block, re.DOTALL)
        # Look for a line-comment immediately before the struct declaration
        pre_window = content[max(0, block_start - 300): block_start]
        desc_m = re.search(r'//\s*(.+?)\s*\n\s*$', pre_window)

        verbs        = re.findall(r'"([^"]+)"', verbs_m.group(1)) if verbs_m else []
        role         = role_m.group(1) if role_m else 'unknown'
        prepositions = re.findall(r'\.(\w+)', prep_m.group(1)) if prep_m else []
        description  = desc_m.group(1).strip() if desc_m else ''

        if verbs:
            results.append({
                'name':         struct_name,
                'verbs':        verbs,
                'role':         role,
                'prepositions': prepositions,
                'description':  description,
            })
    return results


# Scan ALL Swift files under Sources/ARORuntime/Actions/ recursively.
# This catches multi-action files (FileActions.swift, ResponseActions.swift, etc.)
# that were missed by the old per-file single-extraction approach.
actions_dir = ARO_ROOT / 'Sources' / 'ARORuntime' / 'Actions'
all_actions_raw = []
for swift_file in sorted(actions_dir.rglob('*.swift')):
    found = extract_all_actions_from_swift(swift_file)
    if found:
        print(f'  {swift_file.relative_to(ARO_ROOT)}: {len(found)} action(s)')
    all_actions_raw.extend(found)

# Deduplicate: keep first occurrence per primary verb
seen_verbs = set()
actions = []
for a in all_actions_raw:
    key = tuple(a['verbs'])
    if key not in seen_verbs:
        seen_verbs.add(key)
        actions.append(a)

print(f'\nExtracted {len(actions)} unique actions (was 19 before)')
for a in actions[:8]:
    print(f'  {a["name"]}: verbs={a["verbs"][:3]}, role={a["role"]}, preps={a["prepositions"][:3]}')

## Extract Q&A seeds from Proposals

In [ ]:
def extract_from_proposal(md_path):
    content = Path(md_path).read_text()
    name = Path(md_path).stem

    # All ARO code blocks in this proposal
    aro_blocks = re.findall(r'```aro\n(.*?)```', content, re.DOTALL)

    # Split into sections by heading
    section_pattern = re.compile(r'^(#{1,3} .+)', re.MULTILINE)
    parts = section_pattern.split(content)

    qa_seeds = []
    i = 1
    while i < len(parts):
        heading_raw = parts[i]
        body = parts[i + 1] if i + 1 < len(parts) else ''
        heading = re.sub(r'^#+\s*', '', heading_raw).strip()
        section_code = re.findall(r'```aro\n(.*?)```', body, re.DOTALL)

        if len(body.strip()) > 80:
            qa_seeds.append({
                'heading':      heading,
                'body':         body.strip()[:2000],
                'code_examples': [c.strip() for c in section_code],
                'proposal':     name,
            })
        i += 2

    return {
        'name':           name,
        'aro_code_blocks': [b.strip() for b in aro_blocks],
        'qa_seeds':        qa_seeds,
    }

proposals_knowledge = [extract_from_proposal(f) for f in corpus['proposals']]

total_seeds  = sum(len(p['qa_seeds']) for p in proposals_knowledge)
total_blocks = sum(len(p['aro_code_blocks']) for p in proposals_knowledge)
print(f'Proposals: {len(proposals_knowledge)} files, {total_seeds} Q&A seeds, {total_blocks} code blocks')

## Load examples with expected output

In [ ]:
def load_example(example_dir):
    p = Path(example_dir)
    aro_files = {}
    for f in sorted(p.rglob('*.aro')):
        rel = str(f.relative_to(p))
        aro_files[rel] = f.read_text()

    def read_opt(filename):
        fp = p / filename
        return fp.read_text() if fp.exists() else None

    return {
        'name':      p.name,
        'dir':       str(p),
        'aro_files': aro_files,
        'openapi':   read_opt('openapi.yaml'),
        'expected':  read_opt('expected.txt'),
        'readme':    read_opt('README.md'),
        'test_hint': read_opt('test.hint'),
    }

# Discover example root directories
# An example root is a dir that directly contains .aro files or has expected.txt
example_dirs = set()
for f_str in corpus['aro_files']:
    p = Path(f_str)
    # Walk up to find the highest directory that still has .aro files
    # but whose parent is one of the known collection roots
    candidate = p.parent
    while True:
        parent = candidate.parent
        if parent.name in ('Examples', 'ARO-Application', 'sources'):
            break
        if not list(parent.glob('*.aro')) and not (parent / 'expected.txt').exists():
            break
        candidate = parent
    example_dirs.add(str(candidate))

examples = [load_example(d) for d in sorted(example_dirs)]
examples = [e for e in examples if e['aro_files']]

print(f'Loaded {len(examples)} examples')
print(f'  With expected output: {sum(1 for e in examples if e["expected"])}')
print(f'  With openapi.yaml:    {sum(1 for e in examples if e["openapi"])}')

## Save knowledge base

In [ ]:
knowledge = {
    # Generation metadata (issue #382) — correlates knowledge.json with
    # the exact corpus state (commits + manifest) that produced it.
    '_metadata':          build_artifact_metadata(
        num_source_files=(len(corpus['aro_files']) + len(corpus['proposals'])
                          + len(corpus['swift_actions']) + len(corpus['book_chapters'])),
        extra={'artifact': 'knowledge.json',
               'manifest_generated_at': corpus.get('_metadata', {}).get('generated_at')},
    ),
    'actions':            actions,
    'proposals':          proposals_knowledge,
    'examples':           examples,
    'aro_syntax':         corpus['aro_syntax'],
    'aro_actions':        corpus['aro_actions'],
    'extra_docs':         corpus.get('extra_docs', {}),
}

out_path = DATA_OUT / 'knowledge.json'
with open(out_path, 'w') as f:
    json.dump(knowledge, f, indent=2)

print(f'Saved: {out_path}')
print(f'  {len(actions)} actions')
print(f'  {total_seeds} Q&A seeds from proposals')
print(f'  {len(examples)} examples')

In [ ]:
import csv
from pathlib import Path
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_csv_path = _run_dir / '02_knowledge_extraction.csv'

with open(_csv_path, 'w', newline='') as _f:
    _w = csv.writer(_f)
    _w.writerow(['category', 'item_name', 'item_count'])
    _w.writerow(['actions', 'unique_actions', len(actions)])
    _w.writerow(['proposals', 'qa_seeds', total_seeds])
    _w.writerow(['proposals', 'code_blocks', total_blocks])
    _w.writerow(['proposals', 'proposal_files', len(proposals_knowledge)])
    _w.writerow(['examples', 'total_examples', len(examples)])
    _w.writerow(['examples', 'with_expected_output', sum(1 for e in examples if e['expected'])])
    _w.writerow(['examples', 'with_openapi', sum(1 for e in examples if e['openapi'])])

print(f'CSV saved: {_csv_path}  (7 rows)')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222',
    'axes.labelcolor': '#222222',
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'axes.titlecolor': '#111111',
    'legend.edgecolor': '#cccccc',
    'legend.facecolor': 'white',
    'legend.framealpha': 1.0,
    'figure.facecolor': '#fafafa',
    'axes.facecolor': '#f9f9f9',
    'savefig.facecolor': '#fafafa',
    'savefig.bbox': 'tight',
    'savefig.dpi': 150,
})
from pathlib import Path
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '02_knowledge_extraction.png'

_n_actions    = len(actions)
_n_examples   = len(examples)
_total_seeds  = sum(len(p['qa_seeds']) for p in proposals_knowledge)
_total_blocks = sum(len(p['aro_code_blocks']) for p in proposals_knowledge)

_sizes  = [_n_actions, _n_examples, _total_seeds, _total_blocks]
_labels = [f'Actions\n({_n_actions})', f'Examples\n({_n_examples})',
           f'Q&A seeds\n({_total_seeds:,})', f'Code blocks\n({_total_blocks})']
_colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

fig, ax = plt.subplots(figsize=(8, 7))
wedges, _, autotexts = ax.pie(
    _sizes, labels=_labels, colors=_colors, autopct='%1.0f%%',
    startangle=90, explode=[0.04] * 4, pctdistance=0.78,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
)
for t in autotexts:
    t.set_fontsize(10); t.set_fontweight('bold')
# donut hole
ax.add_artist(plt.Circle((0, 0), 0.52, fc='white'))
ax.text(0, 0, f'{sum(_sizes):,}\ntotal', ha='center', va='center',
        fontsize=13, fontweight='bold', color='#2c3e50')
ax.set_title('Knowledge Base Composition', fontsize=13, fontweight='bold', pad=20)
fig.tight_layout()
fig.savefig(_out)
plt.close(fig)
print(f'Saved: {_out}')